# Atom Distance Calculator — `dye2.xyz`
Uses **ASE** to read coordinates and compute Euclidean distances, and **py3Dmol** for interactive 3-D visualisation.

Run cells top-to-bottom. Change the atom indices in **Cell 4** to measure any pair.

In [ ]:
# Cell 1 — install dependencies (run once; skip if already installed)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ase", "py3Dmol"], check=True)
print("Dependencies ready.")

In [1]:
# Cell 2 — imports
import numpy as np
import py3Dmol
from ase.io import read
from IPython.display import display

In [2]:
# Cell 3 — load structure and print atom table
XYZ_FILE = "dye2.xyz"   # adjust path if needed

mol = read(XYZ_FILE)

print(f"{'Idx':>4}  {'Sym':<3}  {'x (Å)':>10}  {'y (Å)':>10}  {'z (Å)':>10}")
print("-" * 46)
for i, atom in enumerate(mol):
    x, y, z = atom.position
    print(f"{i:>4}  {atom.symbol:<3}  {x:>10.5f}  {y:>10.5f}  {z:>10.5f}")

 Idx  Sym       x (Å)       y (Å)       z (Å)
----------------------------------------------
   0  C      -0.25130     0.43141    -1.61453
   1  C      -0.61158    -0.19395    -0.42782
   2  C       1.05634     0.85965    -1.79887
   3  C       2.01253     0.67379    -0.79718
   4  C       1.68124     0.04710     0.44193
   5  C       0.33689    -0.39080     0.58932
   6  C      -0.05344    -1.03075     1.75166
   7  C       0.85568    -1.24724     2.76397
   8  C       2.19549    -0.81788     2.64763
   9  N       2.60003    -0.16392     1.47474
  10  C       4.00919     0.30834     1.34301
  11  C       4.95461    -0.85193     1.01794
  12  C       3.06040    -1.04008     3.69140
  13  C       2.75549    -1.71861     4.99115
  14  C       3.71914    -1.86776     5.92796
  15  C       3.54322    -2.51883     7.28428
  16  C       2.23579    -2.80356     7.73218
  17  C       2.01327    -3.35010     8.98040
  18  C       3.08037    -3.62607     9.81917
  19  C       4.40227    -3.34994

In [8]:
# Cell 3b — 3D view with all atom indices labelled

# Element → label background colour
ELEMENT_COLORS = {
    "C": "#4a4a4a",
    "N": "#3b7ec5",
    "H": "#888888",
    "O": "#c0392b",
    "S": "#c8a000",
}
DEFAULT_COLOR = "#555555"

view = py3Dmol.view(width=800, height=520)
view.addModel(xyz_str, "xyz")   # xyz_str built in Cell 5; re-run Cell 5 first if not defined

# Base style
view.setStyle({}, {"stick":  {"radius": 0.10, "colorscheme": "Jmol"},
                   "sphere": {"radius": 0.25, "colorscheme": "Jmol"}})

# Add an index label at every atom position
for i, atom in enumerate(mol):
    x, y, z = atom.position.tolist()
    color = ELEMENT_COLORS.get(atom.symbol, DEFAULT_COLOR)
    view.addLabel(
        f"{i}:{atom.symbol}",
        {
            "position":          {"x": x, "y": y, "z": z},
            "backgroundColor":   color,
            "fontColor":         "white",
            "fontSize":          16,
            "backgroundOpacity": 0.80,
            "inFront":           True,   # always visible through the molecule
        }
    )

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [3]:
# Cell 4 — *** SET YOUR ATOM INDICES HERE ***
ATOM_A = 9    # e.g. N at index 9
ATOM_B = 24   # e.g. N at index 24

# ── distance calculation (ASE) ──────────────────────────────────────────
pos_a = mol[ATOM_A].position
pos_b = mol[ATOM_B].position
distance = np.linalg.norm(pos_a - pos_b)

sym_a = mol[ATOM_A].symbol
sym_b = mol[ATOM_B].symbol

print(f"Atom A  →  index {ATOM_A:>2}  symbol {sym_a:<2}  position {tuple(round(v,5) for v in pos_a)}")
print(f"Atom B  →  index {ATOM_B:>2}  symbol {sym_b:<2}  position {tuple(round(v,5) for v in pos_b)}")
print()
print(f"Euclidean distance  ({sym_a}{ATOM_A} ↔ {sym_b}{ATOM_B})  =  {distance:.6f} Å")

Atom A  →  index  9  symbol N   position (np.float64(2.60003), np.float64(-0.16392), np.float64(1.47474))
Atom B  →  index 24  symbol N   position (np.float64(4.62349), np.float64(-2.80545), np.float64(8.12025))

Euclidean distance  (N9 ↔ N24)  =  7.432017 Å


In [6]:
# Cell 5 — py3Dmol visualisation
# Rebuild XYZ string from ASE object so the viewer always matches the loaded structure
xyz_lines = [str(len(mol)), "Generated from ASE"]
for atom in mol:
    x, y, z = atom.position
    xyz_lines.append(f"{atom.symbol}  {x:.5f}  {y:.5f}  {z:.5f}")
xyz_str = "\n".join(xyz_lines)

# ── viewer setup ────────────────────────────────────────────────────────
view = py3Dmol.view(width=800, height=500)
view.addModel(xyz_str, "xyz")

# Default style: stick + small sphere
view.setStyle({}, {"stick": {"radius": 0.12, "colorscheme": "Jmol"},
                   "sphere": {"radius": 0.28, "colorscheme": "Jmol"}})

# Highlight atom A (green) and atom B (orange)
view.setStyle({"serial": ATOM_A },   # py3Dmol uses 1-based serials
              {"sphere": {"radius": 0.55, "color": "#3fb950"},
               "stick":  {"radius": 0.18, "color": "#3fb950"}})

view.setStyle({"serial": ATOM_B },
              {"sphere": {"radius": 0.55, "color": "#f0883e"},
               "stick":  {"radius": 0.18, "color": "#f0883e"}})

# Dashed cyan line between the two atoms
ax, ay, az = pos_a.tolist()
bx, by, bz = pos_b.tolist()

view.addCylinder({
    "start": {"x": ax, "y": ay, "z": az},
    "end":   {"x": bx, "y": by, "z": bz},
    "radius": 0.06,
    "fromCap": 1, "toCap": 1,
    "color": "#58a6ff",
    "dashed": True,
    "opacity": 0.9
})

# Label atoms and show distance at midpoint
mid = ((ax + bx) / 2, (ay + by) / 2, (az + bz) / 2)

view.addLabel(f"{sym_a}{ATOM_A}",
              {"position": {"x": ax, "y": ay, "z": az},
               "backgroundColor": "#3fb950", "fontColor": "white",
               "fontSize": 12, "backgroundOpacity": 0.85})

view.addLabel(f"{sym_b}{ATOM_B}",
              {"position": {"x": bx, "y": by, "z": bz},
               "backgroundColor": "#f0883e", "fontColor": "white",
               "fontSize": 12, "backgroundOpacity": 0.85})

view.addLabel(f"{distance:.3f} Å",
              {"position": {"x": mid[0], "y": mid[1], "z": mid[2]},
               "backgroundColor": "#161b22", "fontColor": "#58a6ff",
               "fontSize": 13, "backgroundOpacity": 0.9})

view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
# Cell 6 — measure MULTIPLE pairs at once
# Add as many (i, j) tuples as you like
pairs = [
    (0, 24),   # C0  ↔ N24
    (9, 24),   # N9  ↔ N24
    (0, 51),   # C0  ↔ H51
]

print(f"{'Pair':<14}  {'Sym A':<5}  {'Sym B':<5}  {'Distance (Å)':>14}")
print("-" * 46)
for i, j in pairs:
    d = np.linalg.norm(mol[i].position - mol[j].position)
    label = f"{mol[i].symbol}{i} ↔ {mol[j].symbol}{j}"
    print(f"{label:<14}  {mol[i].symbol:<5}  {mol[j].symbol:<5}  {d:>14.6f}")

Pair            Sym A  Sym B    Distance (Å)
----------------------------------------------
C0 ↔ N24        C      N           11.358115
N9 ↔ N24        N      N            7.432017
C0 ↔ H51        C      H           14.046628
